# Stuttering Detection: Support Vector Machine (SVM) Analysis
**Course**: CS204T (Artificial Intelligence)  
**Team**: 18  
**Focus**: Linear SVM vs. Non-Linear RBF Kernels

---

## Step 1: Initialization & Environment
We use the project's standardized modular engine to load high-dimensional WavLM embeddings. SVMs are particularly powerful for finding optimal 'margins' in high-dimensional spaces like audio features.

In [ ]:
import os
import shutil
import numpy as np
from src.extractors import WavLMExtractor
from src.data import DataManager
from src.models import LinearSVMModel, KernelSVMModel

# Paths to our distributed dataset
AUDIO_DIR = "Stuttering Events in Podcasts Dataset/clips/stuttering-clips/clips"
CSV_PATHS = [
    "Stuttering Events in Podcasts Dataset/SEP-28k_labels.csv",
    "Stuttering Events in Podcasts Dataset/fluencybank_labels.csv"
]
FEATURE_DIR = "data/features"
fluent_dir = os.path.join(FEATURE_DIR, "fluent")
disfluent_dir = os.path.join(FEATURE_DIR, "disfluent")

## Step 2 (Optional): Operational Mode for Data Extraction
Toggle how you want to handle your audio data for this session.
* `SKIP_EXTRACTION`: Uses features already on disk (Default).
* `FORCE_EXTRACT`: Analyzes raw audio for new files (Resumable).
* `CLEAN_START`: Wipes the database and re-extracts from zero.

In [ ]:
# Operational Flags
SKIP_EXTRACTION = True
FORCE_EXTRACT = False
CLEAN_START = False
NUM_CLIPS_TO_EXTRACT = 1000

if CLEAN_START:
    if os.path.exists(FEATURE_DIR):
        shutil.rmtree(FEATURE_DIR)
    print("[System] Clean start initiated. Wiped feature database.")

if not SKIP_EXTRACTION or CLEAN_START or FORCE_EXTRACT:
    extractor = WavLMExtractor("microsoft/wavlm-base")
    label_dict = DataManager.generate_label_dict(CSV_PATHS, filter_quality=True)
    
    # Now using NATIVE Random Sampling logic for diversity
    extractor.extract_from_dir(
        AUDIO_DIR, 
        output_dir=FEATURE_DIR, 
        label_dict=label_dict, 
        limit=NUM_CLIPS_TO_EXTRACT, 
        random_sample=True
    )
else:
    print("[System] Skipping extraction. Using existing data on disk.")

## Step 3: Data Loading & Preprocessing
We load our distributed feature files, perform randomized splitting, and apply **oversampling**. Standardization is essential for SVMs because they rely on the Euclidean distance between vectors.

In [ ]:
label_dict = DataManager.generate_label_dict(CSV_PATHS, filter_quality=True)
manager = DataManager(None, None)

# Loading .npy features into training matrices
X, y = manager.load_from_folders(fluent_dir, disfluent_dir)
X_train, X_val, X_test, y_train, y_val, y_test = manager.get_splits(test_size=0.15, val_size=0.15)

# Handle class imbalance
X_train_bal, y_train_bal = manager.balance_data(X_train, y_train, strategy="oversample")

# Standard Scaling (Fits on training set only!)
X_train_final = manager.preprocess(X_train_bal, method="standard")
X_test_final = manager.preprocess(X_test, method="standard")

## Step 4: Model 1 - Linear SVM
The Linear SVM attempts to find the clearest hyperplane that separates Fluent from Stuttering samples. This is our baseline for structural classification.

In [ ]:
linear_svm = LinearSVMModel("Linear_SVM")
linear_svm.train(X_train_final, y_train_bal)

print("\n--- Evaluation on Unseen Test Set ---")
linear_svm.evaluate(X_test_final, y_test)

## Step 5: Model 2 - Kernel RBF SVM
Real-world audio is rarely perfectly separable. By using the **Radial Basis Function (RBF)** kernel, the model can project data into a higher-dimensional space to find non-linear boundaries. This significantly improves detection for 'tricky' stutters.

In [ ]:
kernel_svm = KernelSVMModel("NonLinear_RBF_SVM", kernel_type='rbf')
kernel_svm.train(X_train_final, y_train_bal)

print("\n--- Evaluation on Unseen Test Set ---")
kernel_svm.evaluate(X_test_final, y_test)